<a href="https://colab.research.google.com/github/dgaida/ba_hcai_th_public/blob/main/assets/exercises/07_tool_calling_minimal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool Calling mit dem LLMClient – ein minimales Beispiel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<GITHUB_USER>/<REPO>/blob/main/<PFAD>/tool_calling_minimal.ipynb)

**Modul:** Mensch-zentrierte Künstliche Intelligenz · **Lernraum 4:** Erweiterte Anwendungen und Reflexion

---

## Worum geht es?

Ein LLM kann nicht zuverlässig rechnen. Wir geben ihm deshalb zwei **Tools** (normale Python-Funktionen):

| Tool | Aufgabe |
|---|---|
| `add` | addiert zwei Zahlen |
| `multiply` | multipliziert zwei Zahlen |

Das LLM **führt die Tools nicht selbst aus**. Es entscheidet nur, *ob* und *mit welchen Argumenten* ein Tool aufgerufen werden soll. Unser Code führt die Funktion aus und gibt das Ergebnis zurück ans LLM.

```
Frage ──► LLM ──► Tool-Aufruf (Name + Argumente)? ──ja──► unser Python-Code ──► Ergebnis ──► LLM ──► Antwort
                          └──────────── nein ────────────────────────────────────────────────────► Antwort
```

Die Entscheidungsgrundlage des LLM sind die **Beschreibungen** der Tools (Name, Zweck, Parameter).
Genau das Prinzip lässt sich später auf euer Semesterprojekt übertragen: `predict_*`-Tool (ML-Modell) oder `search_knowledge`-Tool (RAG).

## Vorbereitung

1. Legt in Google Colab (Schlüssel-Symbol links) ein **Secret** mit dem Namen `API_KEY` an und gebt dort euren API-Key ein.
2. Aktiviert den **Notebook-Zugriff** für dieses Secret (Schalter neben dem Secret).
3. Führt die Zellen der Reihe nach aus.

## Teil 0: Installation und Client erzeugen

In [ ]:
!pip install -q git+https://github.com/dgaida/llm_client.git

In [ ]:
import json

from llm_client import LLMClient

# None = automatische Erkennung anhand des Secrets API_KEY.
# Falls das nicht klappt, den Anbieter explizit angeben, z. B. "groq", "openai" oder "gemini".
API_CHOICE = "groq"

# temperature=0.0: möglichst deterministische Antworten, das erleichtert die Tool-Wahl
client = LLMClient(api_choice=API_CHOICE, temperature=0.0)
print(client)

## Teil 1: Tools definieren

Ein Tool besteht aus zwei Teilen:

1. **der Python-Funktion**, die wirklich etwas tut, und
2. **der Tool-Beschreibung** (JSON-Schema im OpenAI-Format) – nur diese sieht das LLM.

> **Wichtig:** Die `description` ist das Entscheidungskriterium des LLM. Beschreibt darin, *wann* das Tool genutzt werden soll – und wann *nicht*.

In [ ]:
def add(a: float, b: float) -> float:
    """Addiert zwei Zahlen.

    Args:
        a: Erster Summand.
        b: Zweiter Summand.

    Returns:
        Die Summe von a und b.
    """
    return a + b


def multiply(a: float, b: float) -> float:
    """Multipliziert zwei Zahlen.

    Args:
        a: Erster Faktor.
        b: Zweiter Faktor.

    Returns:
        Das Produkt von a und b.
    """
    return a * b


# Zuordnung: Tool-Name (so heißt es für das LLM) -> Python-Funktion
TOOL_FUNCTIONS = {"add": add, "multiply": multiply}

# Beschreibung der Tools für das LLM (OpenAI-Format)
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": (
                "Addiert zwei Zahlen und gibt die Summe zurück. "
                "Nur verwenden, wenn der Nutzer ausdrücklich eine Addition verlangt."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Erster Summand"},
                    "b": {"type": "number", "description": "Zweiter Summand"},
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": (
                "Multipliziert zwei Zahlen und gibt das Produkt zurück. "
                "Nur verwenden, wenn der Nutzer ausdrücklich eine Multiplikation verlangt."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Erster Faktor"},
                    "b": {"type": "number", "description": "Zweiter Faktor"},
                },
                "required": ["a", "b"],
            },
        },
    },
]

## Teil 2: Den Tool-Ablauf implementieren

Der Ablauf in `ask()`:

1. Frage und Tool-Beschreibungen an das LLM schicken (`chat_completion_with_tools`).
2. Hat das LLM **kein** Tool gewählt, ist `result["content"]` bereits die Antwort.
3. Hat das LLM ein Tool gewählt, steht es in `result["tool_calls"]`. Wir **validieren** die Argumente (nicht das LLM!), führen die Funktion aus und
4. schicken das Ergebnis zurück ans LLM, das daraus die Antwort in Alltagssprache formuliert.

> **Hinweis:** Der OpenAI-Standard nutzt für Schritt 4 eine Nachricht mit `role="tool"`. Damit der Code bei allen Anbietern (Groq, OpenAI, Gemini, Ollama) funktioniert, übergeben wir das Tool-Ergebnis hier vereinfacht als normale Nachricht.

In [ ]:
SYSTEM_PROMPT = (
    "Du bist ein hilfsbereiter Assistent. Rechne niemals selbst, sondern nutze für "
    "Additionen und Multiplikationen die bereitgestellten Tools. "
    "Alle anderen Fragen beantwortest du direkt, ohne Tool."
)


def run_tool_call(tool_call: dict) -> str:
    """Führt einen vom LLM vorgeschlagenen Tool-Aufruf aus.

    Name und Argumente werden hier validiert, bevor die Funktion läuft.
    Das LLM kann Tools erfinden oder falsche Argumente liefern.

    Args:
        tool_call: Tool-Aufruf im Format ``{"function": {"name": ..., "arguments": ...}}``.

    Returns:
        Das Ergebnis als Text oder eine Fehlermeldung.
    """
    name = tool_call["function"]["name"]
    arguments = tool_call["function"]["arguments"]
    if isinstance(arguments, str):  # je nach Anbieter String oder dict
        arguments = json.loads(arguments)

    if name not in TOOL_FUNCTIONS:
        return f"Fehler: Unbekanntes Tool '{name}'."
    try:
        a, b = float(arguments["a"]), float(arguments["b"])
    except (KeyError, TypeError, ValueError):
        return f"Fehler: Ungültige Argumente {arguments}."

    result = TOOL_FUNCTIONS[name](a, b)
    print(f"  -> Tool-Aufruf: {name}(a={a}, b={b}) = {result}")
    return str(result)


def ask(question: str) -> str:
    """Stellt dem LLM eine Frage und führt bei Bedarf Tools aus.

    Args:
        question: Frage des Nutzers.

    Returns:
        Die Antwort des LLM.
    """
    print(f"Frage: {question}")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    # Schritt 1: LLM entscheidet, ob und welches Tool es aufrufen möchte
    result = client.chat_completion_with_tools(messages, TOOLS)
    tool_calls = result.get("tool_calls")

    # Schritt 2: kein Tool gewählt -> das LLM hat direkt geantwortet
    if not tool_calls:
        print("  -> Kein Tool aufgerufen.")
        return result["content"]

    # Schritt 3: Tools ausführen (mit Validierung)
    tool_results = [run_tool_call(tc) for tc in tool_calls]

    # Schritt 4: Ergebnisse zurück ans LLM, das die Antwort formuliert
    messages.append(
        {
            "role": "user",
            "content": (
                "Ergebnis des Tool-Aufrufs: " + "; ".join(tool_results) + ". "
                "Beantworte damit die ursprüngliche Frage in einem Satz."
            ),
        }
    )
    return client.chat_completion(messages)

## Teil 3: Drei Beispiele

Beobachtet jeweils die Zeile `-> Tool-Aufruf` bzw. `-> Kein Tool aufgerufen`.

### Beispiel 1: Das LLM soll `add` aufrufen

In [ ]:
print(ask("Was ist 1234 plus 5678?"))

### Beispiel 2: Das LLM soll `multiply` aufrufen

In [ ]:
print(ask("Wie viel ist 23 mal 47?"))

### Beispiel 3: Das LLM soll **kein** Tool aufrufen

In [ ]:
print(ask("Was ist die Hauptstadt von Frankreich?"))

## Aufgaben zum Weiterdenken

1. **Beschreibung ändern:** Streicht in der Beschreibung von `add` den Satz „Nur verwenden, wenn …“ und fragt nach „Wie viel sind drei Äpfel und vier Birnen?“. Ändert sich die Tool-Wahl?
2. **Neues Tool:** Ergänzt ein Tool `subtract` (Funktion, Eintrag in `TOOL_FUNCTIONS` und in `TOOLS`).
3. **Validierung:** Ergänzt in `run_tool_call` eine Prüfung, dass die Zahlen betragsmäßig kleiner als 1 000 000 sind. Was antwortet das LLM auf „Was ist 5 Millionen mal 3 Millionen?“
4. **Transfer auf das Semesterprojekt:** Welche beiden Tools würdet ihr für Projekt B definieren (z. B. `predict_*` und `search_knowledge`)? Schreibt für beide eine `description`, die klar macht, *wann* das jeweilige Tool zu verwenden ist.